# MethylGPT Embedding Extraction
**No external library needed — standalone implementation**

Uses pretrained MethylGPT weights with top-2048 variable CpG probes.

Before running: **Runtime > Change runtime type > T4 GPU > Save**

Then: **Runtime > Run all** — takes ~15-20 minutes for 800 samples.

In [ ]:
# Step 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2 — Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

In [ ]:
# Step 3 — Install minimal dependencies
!pip install -q anndata numpy tqdm

In [ ]:
# Step 4 — Configuration
# UPDATE THIS to your shared Google Drive folder name
SHARED_FOLDER_NAME = "multiomics-project"

from pathlib import Path
DRIVE_ROOT    = Path(f"/content/drive/MyDrive/{SHARED_FOLDER_NAME}")
INPUT_H5AD    = DRIVE_ROOT / "data/processed/tcga_methylation.h5ad"
MODEL_WEIGHTS = DRIVE_ROOT / "pretrained_models/methylgpt-medium/small-best_model_epoch6.pt"
OUTPUT_EMB    = DRIVE_ROOT / "data/processed/methylgpt_embeddings.npy"

N_TOP_PROBES = 2048  # top variable probes (full 49K needs flash_attn, not available in Colab)
BATCH_SIZE   = 16   # safe for T4 16GB with 2048 tokens

print('Methylation h5ad exists :', INPUT_H5AD.exists())
print('Model weights exist     :', MODEL_WEIGHTS.exists())

In [ ]:
# Step 5 — Build MethylGPT model (no library needed)
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.Wqkv     = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, key_padding_mask=None):
        B, T, C = x.shape
        q, k, v = self.Wqkv(x).split(C, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        if key_padding_mask is not None:
            attn = attn.masked_fill(key_padding_mask[:, None, None, :], float('-inf'))
        attn = F.softmax(attn, dim=-1)
        out  = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

class TransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.self_attn = SelfAttention(d_model, n_heads)
        self.linear1   = nn.Linear(d_model, d_model)
        self.linear2   = nn.Linear(d_model, d_model)
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)

    def forward(self, x, key_padding_mask=None):
        x = self.norm1(x + self.self_attn(x, key_padding_mask))
        x = self.norm2(x + self.linear2(F.gelu(self.linear1(x))))
        return x

class TransformerStack(nn.Module):
    def __init__(self, d_model, n_heads, n_layers):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerLayer(d_model, n_heads) for _ in range(n_layers)
        ])

    def forward(self, x, key_padding_mask=None):
        for layer in self.layers:
            x = layer(x, key_padding_mask)
        return x

class ProbeEncoder(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.enc_norm  = nn.LayerNorm(d_model)

    def forward(self, ids):
        return self.enc_norm(self.embedding(ids))

class ValueEncoder(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.linear1 = nn.Linear(1, d_model)
        self.linear2 = nn.Linear(d_model, d_model)
        self.norm    = nn.LayerNorm(d_model)

    def forward(self, v):
        x = F.relu(self.linear1(v.unsqueeze(-1).float()))
        x = F.relu(self.linear2(x))
        return self.norm(x)

class MethylGPT(nn.Module):
    def __init__(self, vocab_size=49159, d_model=128, n_heads=4, n_layers=6):
        super().__init__()
        self.encoder             = ProbeEncoder(vocab_size, d_model)
        self.value_encoder       = ValueEncoder(d_model)
        self.transformer_encoder = TransformerStack(d_model, n_heads, n_layers)

    def forward(self, gene_ids, values, pad_mask=None):
        x = self.encoder(gene_ids) + self.value_encoder(values)
        x = self.transformer_encoder(x, pad_mask)
        return x[:, 0, :]  # CLS token = cell embedding

print('Model architecture defined.')

In [ ]:
# Step 6 — Load pretrained weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = MethylGPT()

state_dict = torch.load(MODEL_WEIGHTS, map_location='cpu')

# Convert fused Wqkv keys and keep only encoder/transformer keys
model_state = {}
for k, v in state_dict.items():
    if not k.startswith(('encoder.', 'value_encoder.', 'transformer_encoder.')):
        continue
    new_k = k.replace('self_attn.Wqkv.weight', 'self_attn.Wqkv.weight')
    model_state[new_k] = v

missing, unexpected = model.load_state_dict(model_state, strict=False)
print(f'Missing keys    : {missing}')
print(f'Unexpected keys : {unexpected[:3] if unexpected else []}')
model.to(device).eval()
print('Pretrained weights loaded!')

In [ ]:
# Step 7 — Load data and select top variable probes
import numpy as np
import anndata as ad

print('Loading methylation data...')
adata = ad.read_h5ad(INPUT_H5AD)
print(f'Shape: {adata.shape}')

# Select top N most variable probes
X = adata.X if isinstance(adata.X, np.ndarray) else adata.X.toarray()
probe_var  = np.var(X, axis=0)
top_idx    = np.argsort(probe_var)[-N_TOP_PROBES:]  # top 2048 by variance
X_selected = X[:, top_idx]  # (800, 2048)

# Token IDs: probes map to IDs 3..49158 (offset by 3 special tokens)
probe_token_ids = top_idx + 3  # (2048,)
CLS_ID = 1

print(f'Selected {N_TOP_PROBES} top variable probes')
print(f'Probe variance range: {probe_var[top_idx].min():.4f} - {probe_var[top_idx].max():.4f}')

In [ ]:
# Step 8 — Extract embeddings
from tqdm import tqdm

n_samples  = X_selected.shape[0]
embeddings = []

# Build gene_ids template: [CLS, probe_0, probe_1, ..., probe_2047]
gene_ids_template = torch.tensor(
    [CLS_ID] + probe_token_ids.tolist(), dtype=torch.long
)  # (2049,)

print(f'Extracting embeddings for {n_samples} samples (batch={BATCH_SIZE})...')

with torch.no_grad():
    for start in tqdm(range(0, n_samples, BATCH_SIZE)):
        end   = min(start + BATCH_SIZE, n_samples)
        batch = X_selected[start:end]  # (B, 2048)
        B     = batch.shape[0]

        # Build values: [-2 for CLS] + beta values
        cls_vals = np.full((B, 1), -2.0, dtype=np.float32)
        values   = np.concatenate([cls_vals, batch], axis=1)  # (B, 2049)

        gene_ids_batch = gene_ids_template.unsqueeze(0).expand(B, -1).to(device)  # (B, 2049)
        values_tensor  = torch.tensor(values, dtype=torch.float32).to(device)

        emb = model(gene_ids_batch, values_tensor)  # (B, 128)
        embeddings.append(emb.cpu().numpy())

        # Save checkpoint every 100 batches
        if (start // BATCH_SIZE + 1) % 100 == 0:
            np.save(OUTPUT_EMB, np.concatenate(embeddings, axis=0))

embeddings = np.concatenate(embeddings, axis=0)
print(f'Done! Shape: {embeddings.shape}')

In [ ]:
# Step 9 — Save to Google Drive
np.save(OUTPUT_EMB, embeddings)
print(f'Saved to : {OUTPUT_EMB}')
print(f'Shape    : {embeddings.shape}')
print(f'Any NaN  : {np.isnan(embeddings).any()}')
print('Done! methylgpt_embeddings.npy is ready.')